# Contrast sensitivity with a vision-language model

Qwen3-VL-2B-Instruct says on which side of the image a faint Gabor is. The Gabors are the
ModelFest ones (Watson & Ahumada, 2005), drawn by `stimuli.py` in this folder:

- **10 spatial frequencies**, 1.12 to 30 cycles/deg (120 px/deg, as in ModelFest)
- **11 contrasts**, 0.03% to 10% (a laptop pilot found the model always right from 1% up)
- **100 trials** per point, 11,000 in all; each trial is a pair of images (Gabor left, Gabor right)

Results go to `../data/model_results.csv`, one row per trial. `analysis.ipynb` makes the numbers
and figures from that file.

Needs a GPU: in Colab, Runtime > Change runtime type > T4 GPU.

## Setup

In [2]:
import os
import sys

inColab = "google.colab" in sys.modules

if inColab:
  from google.colab import drive
  drive.mount("/content/drive")
  os.chdir("/content/drive/MyDrive/Projects/vlm-psychophysics/chapters/contrast_sensitivity/code")
  os.system("pip install -q -U transformers")

sys.path.insert(0, os.getcwd())
print("running in Colab:", inColab)
print("folder:", os.getcwd())

Mounted at /content/drive
running in Colab: True
folder: /content/drive/MyDrive/Projects/vlm-psychophysics/chapters/contrast_sensitivity/code


In [3]:
import time
import torch
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
from transformers       import AutoModelForImageTextToText, AutoProcessor
from transformers.utils import logging

import stimuli

logging.disable_progress_bar()
logging.set_verbosity_error()

# float32 everywhere: the faintest Gabors differ from grey by a tenth of a grey level
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float32
print("device:", device)

device: cuda


In [4]:
modelName = "Qwen/Qwen3-VL-2B-Instruct"

# load the weights straight onto the GPU: going through Colab's RAM first can run out of memory
model = AutoModelForImageTextToText.from_pretrained(modelName, dtype=dtype, device_map=device)
model = model.eval()
proc  = AutoProcessor.from_pretrained(modelName)

question = "Where is the striped pattern: left or right? Answer with one word."

# the words and their token IDs; the model may start with a capital
leftIds  = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ["left", "Left"]]
rightIds = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ["right", "Right"]]

# the question with a blank image: every display has the same size, so the text tokens are
# always the same; only the image's numbers change
blank    = np.full((stimuli.CANVAS_PX[1], stimuli.CANVAS_PX[0]), float(stimuli.MEAN_GREY))
messages = [{"role": "user", "content": [{"type": "image", "image": stimuli.to_image(blank)},
                                         {"type": "text",  "text":  question}]}]
baseInputs = proc.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                      return_dict=True, return_tensors="pt").to(device)
print("model loaded")

model loaded


## Exact contrast, not 8 bits

An ordinary image has 256 grey levels. One level is a contrast of 0.79%, about the human
threshold at 4 c/deg, so the faintest Gabors would vanish. We give the model the exact grey
levels instead: `stimuli.pixel_values` does what the processor does (16 × 16 squares, values
rescaled to -1..1), without rounding to 8 bits. First a check that it matches the processor
on an ordinary 8-bit image.

In [5]:
trial = dict(frequency=4, contrast=0.5, phase=0.0)
grey  = np.round(stimuli.draw_trial(trial, "right"))
theirs = proc.image_processor(images=stimuli.to_image(grey), return_tensors="np")["pixel_values"]
mine   = stimuli.pixel_values(grey)
print("largest difference:", np.abs(theirs - mine).max())

largest difference: 0.0


## The scoring

For each display, how much more the model wants to say "left" than "right" (the log-odds of
its first word). The model leans towards "left", so a trial is a pair of displays: the same
Gabor on the left, then on the right. The trial is correct if the model says "left" more
strongly when the Gabor is on the left. If the two scores are exactly equal, the model saw no difference: the trial counts as a guess
(half right). Guessing gives 50%.

Many displays go through the model at once (a batch); the scores are the same as one at a
time (checked on a laptop: largest difference 0.00001).

In [6]:
def leftOverRight(greys):
  # one score per display; the question is the same for all, only the image numbers change
  n = len(greys)
  inputs = {k: v.repeat(n, *[1] * (v.dim() - 1)) for k, v in baseInputs.items() if k != "pixel_values"}
  inputs["pixel_values"] = torch.from_numpy(np.concatenate([stimuli.pixel_values(g) for g in greys])).to(device, dtype)
  with torch.no_grad():
    scores = model(**inputs, logits_to_keep=1).logits[:, -1].float()
  logp = torch.log_softmax(scores, dim=1)
  return (torch.logsumexp(logp[:, leftIds], 1) - torch.logsumexp(logp[:, rightIds], 1)).tolist()

print("blank display: %.2f (above 0 = prefers left)" % leftOverRight([blank])[0])
for c in [1.0, 0.01]:
  trial = dict(frequency=4, contrast=c, phase=0.0)
  left, right = leftOverRight([stimuli.draw_trial(trial, "left"), stimuli.draw_trial(trial, "right")])
  print("4 c/deg, contrast %g: left %.2f, right %.2f" % (c, left, right))

blank display: 1.18 (above 0 = prefers left)
4 c/deg, contrast 1: left 7.03, right -4.80
4 c/deg, contrast 0.01: left 2.46, right 0.25


## The run

11,000 trials (22,000 images), 8 trials per batch. If Colab disconnects, run the cell again:
trials already in the file are skipped.

In [7]:
resultsFile = "../data/model_results.csv"
batchSize   = 8           # trials per batch (16 images)

if os.path.exists(resultsFile):
  results = pd.read_csv(resultsFile).to_dict("records")
else:
  results = []
done  = set(r["index"] for r in results)
todo  = [t for t in stimuli.make_trials() if t["index"] not in done]
print("%d trials done, %d to go" % (len(done), len(todo)))

startTime = time.time()
lastSave  = len(results)
for b in range(0, len(todo), batchSize):
  batch  = todo[b:b + batchSize]
  greys  = [stimuli.draw_trial(t, side) for t in batch for side in ("left", "right")]
  scores = leftOverRight(greys)

  for k, trial in enumerate(batch):
    left, right = scores[2 * k], scores[2 * k + 1]
    results.append({**trial,
                    "model":       modelName,
                    "score_left":  left,
                    "score_right": right,
                    # a tie: the model saw no difference, so it is a guess (half right)
                    "correct":     1.0 if left > right else 0.5 if left == right else 0.0})
  if len(results) - lastSave >= 500:
    pd.DataFrame(results).to_csv(resultsFile, index=False)
    lastSave = len(results)
    print("%d trials, %.0f s" % (len(results), time.time() - startTime))

pd.DataFrame(results).to_csv(resultsFile, index=False)
print("done: %d trials in %.0f s" % (len(results), time.time() - startTime))

500 trials done, 10500 to go
1004 trials, 259 s
1508 trials, 525 s
2012 trials, 791 s
2516 trials, 1056 s
3020 trials, 1320 s
3524 trials, 1585 s
4028 trials, 1849 s
4532 trials, 2114 s
5036 trials, 2379 s
5540 trials, 2643 s
6044 trials, 2908 s
6548 trials, 3173 s
7052 trials, 3437 s
7556 trials, 3702 s
8060 trials, 3967 s
8564 trials, 4231 s
9068 trials, 4496 s
9572 trials, 4761 s
10076 trials, 5026 s
10580 trials, 5290 s
done: 11000 trials in 5511 s


## Proportion correct

In [8]:
df = pd.DataFrame(results)
table = df.pivot_table(index="frequency", columns="contrast", values="correct")
print("rows: cycles/deg; columns: contrast")
print(table.round(2))

rows: cycles/deg; columns: contrast
contrast  0.00032 0.00056 0.00100 0.00178 0.00316 0.00562 0.01000 0.01778  \
frequency                                                                   
1.12          0.0     0.0    0.04     0.0    0.19    0.58     1.0     1.0   
2.00         0.44    0.45    0.27    0.05    0.54     1.0     1.0     1.0   
2.83         0.39    0.45    0.49    0.41    0.99     1.0     1.0     1.0   
4.00         0.45    0.41    0.24     0.2    0.96     1.0     1.0     1.0   
5.66         0.18     0.2    0.24    0.39     1.0     1.0     1.0     1.0   
8.00         0.53    0.45    0.33     0.1    0.38     1.0     1.0     1.0   
11.30        0.44    0.47    0.22    0.39    0.55     1.0     1.0     1.0   
16.00        0.48    0.48    0.37    0.08     0.4     1.0     1.0     1.0   
22.60        0.28    0.43     0.4    0.38     0.0    0.88     1.0     1.0   
30.00        0.38     0.1     0.0    0.21     0.0    0.18     1.0     1.0   

contrast  0.03162 0.05623 0.10000  
fre